In [1]:
import warnings
warnings.simplefilter("ignore")
import numpy as np
import pandas as pd 
from lightkurve import LightCurve
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve 
import lightkurve as lk

for att in ['axes.labelsize', 'axes.titlesize', 'legend.fontsize',
            'legend.fontsize', 'xtick.labelsize', 'ytick.labelsize']:
    plt.rcParams[att] = 15

search_result = search_lightcurve("AU Mic") 
#print(search_result)
lc2min = search_result[4].download() 
lc2 = lc2min.remove_nans().remove_outliers()
lc2 = lc2[lc2.quality == 0]
lc2_m = lc2.normalize().remove_nans()

x2min = np.ascontiguousarray(lc2_m.time.value, dtype=np.float64) 
y2min = np.ascontiguousarray(lc2_m.flux, dtype=np.float64)
yerr2min = np.ascontiguousarray(lc2_m.flux_err, dtype=np.float64)
lcquality = np.ascontiguousarray(lc2_m.quality, dtype=np.float64)

In [3]:

# ============================================================
# PASSO 2: CARREGAR MÁSCARA DE FLARES E TRÂNSITOS (CSV)
# ============================================================
print("\n" + "="*60)
print("ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do CSV")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# Arquivo com colunas: t_ini, t_fim
csv_path = "intervalos_flares_transitos.csv"

try:
    df_intervalos = pd.read_csv(csv_path)

    # Validação básica das colunas esperadas
    colunas_esperadas = {"t_ini", "t_fim"}
    if not colunas_esperadas.issubset(df_intervalos.columns):
        raise ValueError(f"CSV deve conter as colunas {colunas_esperadas}. Colunas encontradas: {set(df_intervalos.columns)}")

    # Converte para lista de pares [inicio, fim]
    mascara_flares_list = df_intervalos[["t_ini", "t_fim"]].dropna().values.tolist()

    print(f"✓ {len(mascara_flares_list)} intervalo(s) carregado(s) de '{csv_path}':")
    for i, (ini, fim) in enumerate(mascara_flares_list, start=1):
        print(f"  {i}. [{ini:.6f}, {fim:.6f}]")

except Exception as e:
    print(f"✗ Erro ao carregar '{csv_path}': {e}")
    print("→ Usando lista vazia.")
    mascara_flares_list = []

# Criar máscara booleana
mascara_flares = np.zeros(len(t), dtype=bool)
for ini, fim in mascara_flares_list:
    mascara_flares |= (t >= ini) & (t <= fim)

mask_good_flares = ~mascara_flares

print(f"\nMáscara criada:")
print(f"  Pontos excluídos (flares/trânsitos): {mascara_flares.sum()}")
print(f"  Pontos para ajuste: {mask_good_flares.sum()}")



ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do CSV
✓ 29 intervalo(s) carregado(s) de 'intervalos_flares_transitos.csv':
  1. [3883.239461, 3883.334868]
  2. [3884.751868, 3884.843937]
  3. [3885.024950, 3885.109542]
  4. [3885.549959, 3885.617899]
  5. [3885.738284, 3885.759441]
  6. [3886.130183, 3886.411659]
  7. [3888.081298, 3888.115210]
  8. [3888.784350, 3888.805991]
  9. [3889.179949, 3889.251286]
  10. [3889.826802, 3889.854535]
  11. [3890.321862, 3890.405131]
  12. [3891.013277, 3891.076262]
  13. [3891.741181, 3891.795356]
  14. [3891.844004, 3891.908792]
  15. [3891.950028, 3892.001562]
  16. [3892.631057, 3892.697367]
  17. [3892.812488, 3892.900441]
  18. [3893.017865, 3893.101673]
  19. [3893.403291, 3893.458281]
  20. [3896.687407, 3896.849926]
  21. [3897.567644, 3897.779646]
  22. [3899.153300, 3899.218797]
  23. [3899.542373, 3899.616046]
  24. [3900.250646, 3900.422185]
  25. [3900.680283, 3900.764591]
  26. [3903.026735, 3903.311503]
  27. [3904.512267, 3

In [19]:
from astropy.stats import sigma_clip
import numpy as np
import matplotlib.pyplot as plt
from lightkurve import LightCurve

# ============================================================
# PASSO 3: AJUSTE POLINOMIAL + FLATTEN LOCAL
# ============================================================

print("\n" + "="*60)
print("AJUSTE: Polinomial com Segmentos e Costura de Bordas")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# ============================================================
# REGIÕES MANUAIS
# ============================================================

ajustes_manuais = [

    [3883.0050, 3883.4554, "poly",    4, 2.5],
    [3883.1409, 3883.7849, "poly",    4, 2.5],
    [3883.8264, 3884.0955, "poly",    4, 2.5],
    [3884.2828, 3885.3000, "poly",    4, 2.5],
    [3885.2839, 3885.4326, "poly",    4, 2.5],
    [3885.6205, 3885.7794, "poly",    2, 2.5],
    
    [3892.6280, 3892.7040, "poly",    1, 2.5],


    #[3887.1853, 3888.4396, "flatten", None, None],
]

# ============================================================
# FLATTEN GLOBAL
# ============================================================

flcd, trend = lc2_m.flatten(
    window_length=320,
    polyorder=3,
    return_trend=True,
    break_tolerance=10,
    niters=4,
    sigma=2.5,
    mask=mask_good_flares
)

# ============================================================
# AJUSTE AUTOMÁTICO
# ============================================================

N_SEGMENTOS_AUTO = 40
GRAU_AUTO = 4
SIGMA_AUTO = 2.9

# ============================================================
# FUNÇÕES
# ============================================================

def fitting_segment(t_seg, f_seg, mask_seg, deg, sigma):

    if len(t_seg) < deg + 2:
        return np.full_like(f_seg, np.nanmedian(f_seg))

    t_mid = np.median(t_seg)
    t_s = t_seg - t_mid

    good = mask_seg.copy()

    modelo = np.full_like(f_seg, np.nan)

    for _ in range(5):

        if np.sum(good) < deg + 2:
            break

        coef = np.polyfit(
            t_s[good],
            f_seg[good],
            deg=deg
        )

        modelo_good = np.polyval(
            coef,
            t_s[good]
        )

        modelo = np.interp(
            t_s,
            t_s[good],
            modelo_good
        )

        resid = f_seg - modelo

        clipped = sigma_clip(
            resid[good],
            sigma=sigma,
            maxiters=1
        )

        if clipped.mask is np.ma.nomask:
            break

        good[np.where(good)[0]] = ~clipped.mask

    if np.all(np.isnan(modelo)):
        modelo = np.full_like(
            f_seg,
            np.nanmedian(f_seg)
        )

    return modelo


def costurar_bordas(
    t,
    modelo,
    bordas,
    tamanho_janela=10
):

    modelo_costurado = np.copy(modelo)

    print(
        f"\nAplicando costura em "
        f"{len(bordas)} bordas..."
    )

    for borda_idx in bordas:

        inicio = max(
            0,
            borda_idx - tamanho_janela
        )

        fim = min(
            len(t)-1,
            borda_idx + tamanho_janela
        )

        if fim <= inicio:
            continue

        fluxo_A = modelo[inicio]
        fluxo_B = modelo[fim]

        tempo_A = t[inicio]
        tempo_B = t[fim]

        tempos = t[inicio:fim+1]

        transicao = np.interp(
            tempos,
            [tempo_A, tempo_B],
            [fluxo_A, fluxo_B]
        )

        modelo_costurado[inicio:fim+1] = transicao

    return modelo_costurado

# ============================================================
# AJUSTE AUTOMÁTICO BASE
# ============================================================

modelo_manchas = np.zeros_like(f)

bordas_indices = set()

edges = np.linspace(
    t.min(),
    t.max(),
    N_SEGMENTOS_AUTO + 1
)

segmentos = [
    (edges[i], edges[i+1])
    for i in range(N_SEGMENTOS_AUTO)
]

print("Ajuste automático...")

for i, (ini, fim) in enumerate(segmentos):

    idx = (t >= ini) & (t <= fim)

    if not np.any(idx):
        continue

    modelo_manchas[idx] = fitting_segment(
        t[idx],
        f[idx],
        mask_good_flares[idx],
        GRAU_AUTO,
        SIGMA_AUTO
    )

    if i > 0:
        bordas_indices.add(
            np.where(idx)[0][0]
        )

# ============================================================
# AJUSTES MANUAIS
# ============================================================

print(
    f"Aplicando "
    f"{len(ajustes_manuais)} ajustes manuais..."
)

for ini, fim, metodo, grau, sig in ajustes_manuais:

    idx = (t >= ini) & (t <= fim)

    if not np.any(idx):
        continue

    if metodo == "poly":

        modelo_manchas[idx] = fitting_segment(
            t[idx],
            f[idx],
            mask_good_flares[idx],
            grau,
            sig
        )

    elif metodo == "flatten":

        print(
            f"USANDO FLATTEN "
            f"{ini:.4f} - {fim:.4f}"
        )

        modelo_manchas[idx] = trend.flux.value[idx]

        idx_inicio = np.where(idx)[0][0]

        janela = 15

        i0 = max(
            0,
            idx_inicio - janela
        )

        i1 = min(
            len(modelo_manchas)-1,
            idx_inicio + janela
        )

        modelo_manchas[i0:i1+1] = np.linspace(
            modelo_manchas[i0],
            modelo_manchas[i1],
            i1 - i0 + 1
        )

    bordas_indices.add(
        np.where(idx)[0][0]
    )

    bordas_indices.add(
        np.where(idx)[0][-1]
    )

# ============================================================
# COSTURA FINAL
# ============================================================

modelo_manchas_suave = costurar_bordas(
    t,
    modelo_manchas,
    sorted(bordas_indices),
    tamanho_janela=10
)

# ============================================================
# INSPEÇÃO DO TRECHO PROBLEMÁTICO
# ============================================================

idx_print = (
    (t >= 3887.15) &
    (t <= 3887.25)
)

print("\nTRECHO 3887.15 - 3887.25\n")

for tempo, valor in zip(
    t[idx_print],
    modelo_manchas_suave[idx_print]
):
    print(
        f"{tempo:.10f}    "
        f"{valor:.10f}"
    )

# ============================================================
# RESÍDUO
# ============================================================

residual_manchas = (
    f /
    modelo_manchas_suave
)

print("\nOK Ajuste concluído.")

# ============================================================
# GRÁFICOS
# ============================================================

%matplotlib qt

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(12,10)
)

ax1.plot(
    t,
    f,
    'k.-',
    ms=1.5,
    lw=0.5,
    alpha=0.6,
    label='Dados'
)

ax1.plot(
    t,
    modelo_manchas_suave,
    'r-',
    lw=2.5,
    label='Modelo'
)

ax1.set_ylabel("Fluxo")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(
    t,
    residual_manchas,
    'b.-',
    ms=1.5,
    lw=0.5,
    alpha=0.7
)

ax2.axhline(
    1,
    ls='--',
    alpha=0.5
)

ax2.set_ylabel("Residual")
ax2.set_xlabel("Tempo [BTJD]")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


AJUSTE: Polinomial com Segmentos e Costura de Bordas
Ajuste automático...
Aplicando 7 ajustes manuais...

Aplicando costura em 50 bordas...

TRECHO 3887.15 - 3887.25

3887.1512388559    1.0019648075
3887.1526277395    1.0020159483
3887.1540166231    1.0020672083
3887.1554055071    1.0021185875
3887.1567943907    1.0021698475
3887.1581832742    1.0022212267
3887.1595721583    1.0022726059
3887.1609610418    1.0023241043
3887.1623499254    1.0023754835
3887.1637388094    1.0024269819
3887.1651276930    1.0024784803
3887.1665165766    1.0025299788
3887.1679054601    1.0025815964
3887.1692943442    1.0026332140
3887.1706832277    1.0026848316
3887.1720721113    1.0027364492
3887.1734609949    1.0027881861
3887.1748498785    1.0028399229
3887.1762387625    1.0028916597
3887.1776276461    1.0029433966
3887.1790165296    1.0029952526
3887.1804054132    1.0030469894
3887.1817942968    1.0030988455
3887.1831831803    1.0031508207
3887.1845720639    1.0032026768
3887.1859609479    1.0032546520


In [12]:
from astropy.stats import sigma_clip
import numpy as np
import matplotlib.pyplot as plt
from lightkurve import LightCurve

# ============================================================
# PASSO 3: AJUSTE POLINOMIAL COM COSTURA DE BORDAS
# ============================================================
print("\n" + "="*60)
print("AJUSTE: Polinomial com Segmentos e Costura de Bordas")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# --- 1) AJUSTES MANUAIS (usando a versão corrigida que propus) ---
ajustes_manuais = [
    [3883.005, 3883.4554, 4, 2.5],
    [3883.1409, 3883.7849, 4, 2.5],
    [3883.8264, 3883.0955, 4, 2.5],
    [3884.2828, 3885.3000, 4, 2.5],
    [3885.2839, 3885.4326, 4, 2.5],
    [3885.6205, 3885.7794, 4, 2.5],

    [3887.1853, 3888.4396, 4, 1.4],
]

# Flatten com return_trend=True
flcd, trend = lc2_m.flatten(
    window_length=320,
    polyorder=3,
    return_trend=True,
    break_tolerance=10,
    niters=4,
    sigma=6.5,
    mask=mask_good_flares
)
# --- 2) CONFIGURACAO DO AJUSTE AUTOMATICO ---
N_SEGMENTOS_AUTO = 10
GRAU_AUTO = 4
SIGMA_AUTO = 2.9

# ============================================================
# FUNÇÕES DE AJUSTE E COSTURA
# ============================================================

def fitting_segment(t_seg, f_seg, mask_seg, deg, sigma):
    """Ajusta polinomio em um segmento (função original)."""
    if len(t_seg) < deg + 2: return np.full_like(f_seg, np.nanmedian(f_seg))
    t_mid = np.median(t_seg); t_s = t_seg - t_mid
    good = mask_seg.copy()
    modelo = np.full_like(f_seg, np.nan)
    for _ in range(5):
        if np.sum(good) < (deg + 2):
            break
        coef = np.polyfit(t_s[good], f_seg[good], deg=deg)
        
        # 1. Calcula o polinômio APENAS nos pontos válidos
        modelo_good = np.polyval(coef, t_s[good])
        
        # 2. Usa interpolação linear para preencher os "buracos" dos flares/trânsitos
        modelo = np.interp(t_s, t_s[good], modelo_good)
        
        resid = f_seg - modelo
        clipped = sigma_clip(resid[good], sigma=sigma, maxiters=1)
        if clipped.mask is np.ma.nomask:
            break
        good[np.where(good)[0]] = ~clipped.mask
    if np.all(np.isnan(modelo)): modelo = np.full_like(f_seg, np.nanmedian(f_seg))
    return modelo




def costurar_bordas(t, modelo, bordas, tamanho_janela=10):
    """Aplica uma transição linear suave nas bordas dos segmentos."""
    modelo_costurado = np.copy(modelo)
    print(f"\nAplicando costura em {len(bordas)} bordas com janela de {tamanho_janela} pontos...")
    
    for borda_idx in bordas:
        # Define a janela de costura ao redor da borda
        inicio_janela = max(0, borda_idx - tamanho_janela)
        fim_janela = min(len(t) - 1, borda_idx + tamanho_janela)
        
        if fim_janela <= inicio_janela: continue

        # Pontos de ancoragem para a linha reta
        ponto_A_idx = inicio_janela
        ponto_B_idx = fim_janela
        
        tempo_A, fluxo_A = t[ponto_A_idx], modelo[ponto_A_idx]
        tempo_B, fluxo_B = t[ponto_B_idx], modelo[ponto_B_idx]

        # Cria a linha reta (interpolação linear)
        tempos_janela = t[ponto_A_idx:ponto_B_idx+1]
        fluxos_costura = np.interp(tempos_janela, [tempo_A, tempo_B], [fluxo_A, fluxo_B])
        
        # Substitui o modelo original pela costura na janela
        modelo_costurado[ponto_A_idx:ponto_B_idx+1] = fluxos_costura
        
    return modelo_costurado

# ============================================================
# EXECUÇÃO DO AJUSTE
# ============================================================

modelo_manchas = np.zeros_like(f)
bordas_indices = set()

# A) Ajuste automatico base
edges = np.linspace(t.min(), t.max(), N_SEGMENTOS_AUTO + 1)
segmentos = [(edges[i], edges[i + 1]) for i in range(N_SEGMENTOS_AUTO)]
print("Ajuste automatico em 10 segmentos...")
for i, (ini, fim) in enumerate(segmentos):
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx): continue
    modelo_manchas[idx] = fitting_segment(t[idx], f[idx], mask_good_flares[idx], GRAU_AUTO, SIGMA_AUTO)
    # Guarda o índice do início de cada segmento (exceto o primeiro)
    if i > 0:
        borda_idx = np.where(idx)[0][0]
        bordas_indices.add(borda_idx)

# B) Ajustes manuais (sobrescreve)
print(f"Aplicando {len(ajustes_manuais)} ajustes manuais...")
for ini, fim, grau, sig in ajustes_manuais:

    idx = (t >= ini) & (t <= fim)
    if not np.any(idx):
        continue

    if np.isclose(ini, 3887.1853):

        print("USANDO FLATTEN:", ini, fim)

        modelo_manchas[idx] = trend.flux.value[idx]

    else:

        modelo_manchas[idx] = fitting_segment(
            t[idx],
            f[idx],
            mask_good_flares[idx],
            grau,
            sig
        )

    bordas_indices.add(np.where(idx)[0][0])
    bordas_indices.add(np.where(idx)[0][-1])

# C) Costura das Bordas
# C) Costura das Bordas
idx_print = (t >= 3887.15) & (t <= 3887.25)

for tempo, modelo in zip(t[idx_print], modelo_manchas[idx_print]):
    print(f"{tempo:.10f}  {modelo:.10f}")

modelo_manchas_suave = costurar_bordas(
    t,
    modelo_manchas,
    sorted(list(bordas_indices)),
    tamanho_janela=10
)

# ============================================================
# PRINT DO MODELO ENTRE 3887.15 E 3887.25
# ============================================================

idx_print = (t >= 3887.15) & (t <= 3887.25)

with open("modelo_3887.15_3887.25.txt", "w") as arq:
    arq.write("Tempo\tModelo\n")
    for tempo, modelo in zip(t[idx_print], modelo_manchas_suave[idx_print]):
        arq.write(f"{tempo:.10f}\t{modelo:.10f}\n")

print("Arquivo salvo: modelo_3887.15_3887.25.txt")

# Residuo final
residual_manchas = f / modelo_manchas_suave
print("\nOK Ajuste e costura concluídos.")
# Residuo final
residual_manchas = f / modelo_manchas_suave
print("\nOK Ajuste e costura concluídos.")

# ============================================================
# GRÁFICO DE INSPEÇÃO
# ============================================================
%matplotlib qt
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados Originais')
ax1.plot(t, modelo_manchas_suave, 'r-', linewidth=2.5, label='Ajuste Suavizado')
#ax1.plot(t, modelo_manchas, 'g--', linewidth=1.5, alpha=0.5, label='Ajuste Original (com degraus)') # Descomente para comparar
ax1.set_ylabel('Fluxo Normalizado')
ax1.set_title('Passo 1: Ajuste Polinomial com Transição Suave', fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.7, label='Residual')
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylabel('Fluxo Residual')
ax2.set_xlabel('Tempo [BTJD dias]')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


AJUSTE: Polinomial com Segmentos e Costura de Bordas
Ajuste automatico em 10 segmentos...
Aplicando 7 ajustes manuais...
USANDO FLATTEN: 3887.1853 3888.4396
3887.1512388559  1.0020685196
3887.1526277395  1.0021208525
3887.1540166231  1.0021731853
3887.1554055071  1.0022255182
3887.1567943907  1.0022779703
3887.1581832742  1.0023304224
3887.1595721583  1.0023828745
3887.1609610418  1.0024354458
3887.1623499254  1.0024880171
3887.1637388094  1.0025405884
3887.1651276930  1.0025932789
3887.1665165766  1.0026459694
3887.1679054601  1.0026987791
3887.1692943442  1.0027515888
3887.1706832277  1.0028043985
3887.1720721113  1.0028573275
3887.1734609949  1.0029102564
3887.1748498785  1.0029633045
3887.1762387625  1.0030163527
3887.1776276461  1.0030694008
3887.1790165296  1.0031225681
3887.1804054132  1.0031757355
3887.1817942968  1.0032289028
3887.1831831803  1.0032821894
3887.1845720639  1.0033354759
3887.1859609479  1.0060722828
3887.1873498315  1.0061339140
3887.1887387151  1.0061955452
38

In [14]:




from astropy.stats import sigma_clip
import numpy as np
import matplotlib.pyplot as plt
from lightkurve import LightCurve

# ============================================================
# PASSO 3: SEGMENTACAO (AUTO + MANUAL) E AJUSTE POLINOMIAL
# ============================================================
print("\n" + "="*60)
print("AJUSTE: Polinomial com Segmentos Automaticos + Manuais")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# --- 1) AJUSTES MANUAIS DO POLINOMIO (edite aqui) ---
# Formato: [tempo_inicio, tempo_fim, grau_polinomio, sigma_clipping]

ajustes_manuais = [
    [3883.005, 3883.4554, 4, 1.5],       # ✓ OK - mantém original
    [3883.1409, 3883.7849, 4, 2.5],      # ✓ Corrigido: inverteu a ordem, sigma reduzido
    [3884.2828, 3884.6983, 3, 3.5],      # ✓ Sigma reduzido de 6.5 → 3.5
    [3884.6650, 3885.1138, 4, 2.5],      # ✓ Consolidado de 2 blocos em 1 | Sigma reduzido
    [3885.1191, 3885.2983, 4, 2.5],      # ✓ Consolidado de 2 blocos em 1 | Sigma reduzido

]
# --- 2) CONFIGURACAO DO AJUSTE AUTOMATICO (restante da curva) ---
N_SEGMENTOS_AUTO = 10
GRAU_AUTO = 4
SIGMA_AUTO = 3.0

# --- 3) AJUSTE LOCAL TIPO FLATTEN POR REGIAO MANUAL (opcional) ---
# Se quiser controlar no estilo xdefinitivo + mascara + flatten, edite esta lista.
# IMPORTANTE: os intervalos do CSV sempre serao excluidos no ajuste.
# mascara_excluir adiciona mascaras extras (alem da mascara do CSV).
ajustes_flatten_manuais = [
    {
        "intervalo": [2036.8961, 2037.0935],
        "mascara_excluir": [[2036.9702, 2037.0661]],
        "window_length": 50,
        "polyorder": 1,
        "break_tolerance": 10,
        "niters": 2,
        "sigma": 2.5,
    },
    {
        "intervalo": [2047.4854, 2047.6920],
        "mascara_excluir": [[2047.6266, 2047.6796]],
        "window_length": 30,
        "polyorder": 1,
        "break_tolerance": 10,
        "niters": 2,
        "sigma": 2.5,
    },
    {
        "intervalo": [2058.7802, 2058.8853],
        "mascara_excluir": [[2058.8314, 2058.8763]],
        "window_length": 30,
        "polyorder": 1,
        "break_tolerance": 10,
        "niters": 4,
        "sigma": 2.5,
    },
    {
        "intervalo": [2059.0268, 2059.1702],
        "mascara_excluir": [[2059.0928, 2059.1565]],
        "window_length": 35,
        "polyorder": 1,
        "break_tolerance": 7,
        "niters": 1,
        "sigma": 2.5,
    },
]

modelo_manchas = np.zeros_like(f, dtype=float)

def fitting_segment(t_seg, f_seg, mask_seg, deg, sigma):
    """Ajusta polinomio em um segmento ignorando regioes de flare/transito."""
    if len(t_seg) == 0:
        return np.zeros_like(f_seg, dtype=float)

    t_mid = np.median(t_seg)
    t_s = t_seg - t_mid
    good = mask_seg.copy()

    modelo = np.full_like(f_seg, np.nan, dtype=float)

    for _ in range(5):
        if np.sum(good) < (deg + 2):
            break
        coef = np.polyfit(t_s[good], f_seg[good], deg=deg)
        modelo = np.polyval(coef, t_s)
        resid = f_seg - modelo
        clipped = sigma_clip(resid[good], sigma=sigma, maxiters=1)
        if clipped.mask is np.ma.nomask:
            break
        good[np.where(good)[0]] = ~clipped.mask

    if np.all(np.isnan(modelo)):
        modelo = np.full_like(f_seg, np.nanmedian(f_seg), dtype=float)

    return modelo

# A) Ajuste automatico base (10 segmentos de tempo iguais)
edges = np.linspace(t.min(), t.max(), N_SEGMENTOS_AUTO + 1)
segmentos = [(edges[i], edges[i + 1]) for i in range(N_SEGMENTOS_AUTO)]
print(f"Ajuste automatico: {N_SEGMENTOS_AUTO} segmentos de tempo iguais.")
print("Mascara usada no ajuste: intervalos do CSV (sempre excluidos).")

for i, (ini, fim) in enumerate(segmentos):
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx):
        continue
    modelo_manchas[idx] = fitting_segment(
        t[idx], f[idx], mask_good_flares[idx], GRAU_AUTO, SIGMA_AUTO
    )

# B) Ajustes manuais (sobrescreve regioes escolhidas)
if len(ajustes_manuais) > 0:
    print(f"Aplicando {len(ajustes_manuais)} ajuste(s) manual(is):")
    for ini, fim, grau, sig in ajustes_manuais:
        if fim < ini:
            print(f"  [AVISO] Intervalo invertido em [{ini}, {fim}]. Corrigindo a ordem.")
            ini, fim = fim, ini
        idx = (t >= ini) & (t <= fim)
        if not np.any(idx):
            print(f"  [AVISO] Nenhum ponto em [{ini}, {fim}] - ignorado.")
            continue
        modelo_manchas[idx] = fitting_segment(
            t[idx], f[idx], mask_good_flares[idx], grau, sig
        )
        print(f"  OK [{ini}, {fim}] -> grau={grau}, sigma={sig} ({idx.sum()} pontos)")

# Residuo final
residual_manchas = f / modelo_manchas
print("\nOK Ajuste concluido. Graficos de inspecao abaixo.")

# ============================================================
# GRAFICO 1 (global): igual ao seu modelo de 2 paineis
# ============================================================
%matplotlib qt
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados Originais')
ax1.plot(t, modelo_manchas, 'r-', linewidth=2.5, label='Ajuste (Manchas)')
for m_ini, m_fim in mascara_flares_list:
    ax1.axvspan(m_ini, m_fim, color='red', alpha=0.10)
ax1.set_ylabel('Fluxo Normalizado', fontsize=14)
ax1.set_title('Passo 1: Dados Originais + Ajuste', fontsize=15, fontweight='bold')
ax1.legend(loc='upper right', fontsize=12)
ax1.grid(alpha=0.3)

ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.7, label='Residual')
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
for m_ini, m_fim in mascara_flares_list:
    ax2.axvspan(m_ini, m_fim, color='red', alpha=0.10)
ax2.set_ylabel('Fluxo Residual', fontsize=14)
ax2.set_xlabel('Tempo [BTJD dias]', fontsize=14)
ax2.set_title('Residuais para selecao de flares/transitos', fontsize=15, fontweight='bold')
ax2.legend(loc='upper right', fontsize=12)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

"""

# ============================================================
# GRAFICO 2: inspecao individual dos segmentos automaticos
# ============================================================
fig_auto, axs_auto = plt.subplots(2, 5, figsize=(18, 7), sharey=True)
axs_auto = axs_auto.ravel()

for i, (ini, fim) in enumerate(segmentos):
    ax = axs_auto[i]
    idx = (t >= ini) & (t <= fim)
    ax.plot(t[idx], f[idx], 'k.', ms=2, alpha=0.6)
    ax.plot(t[idx], modelo_manchas[idx], 'r-', lw=1.8)
    for m_ini, m_fim in mascara_flares_list:
        if (m_fim >= ini) and (m_ini <= fim):
            ax.axvspan(max(m_ini, ini), min(m_fim, fim), color='red', alpha=0.12)
    ax.set_title(f'Seg {i+1}\n[{ini:.3f}, {fim:.3f}]', fontsize=9)
    ax.grid(alpha=0.25)

fig_auto.suptitle('Inspecao por segmento automatico', fontsize=13)
plt.tight_layout()
plt.show()
"""

"""


# ============================================================
# GRAFICO 3: ajustes manuais estilo recorte + flatten por regiao
# ============================================================
manual_flatten_results = []

for k, cfg in enumerate(ajustes_flatten_manuais, start=1):
    ini, fim = cfg["intervalo"]
    idx_reg = (x2min >= ini) & (x2min <= fim)

    if not np.any(idx_reg):
        print(f"[AVISO] Ajuste local {k}: sem pontos no intervalo [{ini}, {fim}].")
        continue

    x_local = np.array(x2min[idx_reg], dtype=np.float64)
    y_local = np.array(y2min[idx_reg], dtype=np.float64)

    mascara_local = np.zeros(len(x_local), dtype=bool)

    # Mascara base obrigatoria: intervalos do CSV que cruzam a regiao
    intervalos_csv_regiao = []
    for m_ini, m_fim in mascara_flares_list:
        if (m_fim >= ini) and (m_ini <= fim):
            intervalos_csv_regiao.append([m_ini, m_fim])

    # Mascara extra opcional definida manualmente
    intervalos_extras = cfg.get("mascara_excluir", [])

    # Combinacao: CSV (sempre) + extras
    intervalos_mascara = intervalos_csv_regiao + intervalos_extras

    for m_ini, m_fim in intervalos_mascara:
        mascara_local |= (x_local >= m_ini) & (x_local <= m_fim)

    lc_local = LightCurve(time=x_local, flux=y_local)
    flcd_local = lc_local.flatten(
        window_length=cfg.get("window_length", 30),
        polyorder=cfg.get("polyorder", 1),
        return_trend=False,
        break_tolerance=cfg.get("break_tolerance", 10),
        niters=cfg.get("niters", 2),
        sigma=cfg.get("sigma", 2.5),
        mask=mascara_local,
    )

    tempo_local_det = np.ascontiguousarray(flcd_local.time.value, dtype=np.float64)
    fluxo_local_det = np.ascontiguousarray(flcd_local.flux, dtype=np.float64)

    manual_flatten_results.append({
        "intervalo": [ini, fim],
        "tempo": tempo_local_det,
        "fluxo": fluxo_local_det,
        "mask": mascara_local,
        "intervalos_csv": intervalos_csv_regiao,
        "intervalos_extras": intervalos_extras,
    })

    # Plot local: bruto e flatten
    fig_loc, (axl1, axl2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

    idx_t = (t >= ini) & (t <= fim)
    axl1.plot(t[idx_t], f[idx_t], 'k.-', ms=3, lw=0.8, alpha=0.65, label='Original (recorte)')
    axl1.plot(t[idx_t], modelo_manchas[idx_t], 'r-', lw=2.0, label='Polinomio aplicado')

    for m_ini, m_fim in intervalos_csv_regiao:
        axl1.axvspan(m_ini, m_fim, color='red', alpha=0.18, label='Mascara CSV')
    for m_ini, m_fim in intervalos_extras:
        axl1.axvspan(m_ini, m_fim, color='orange', alpha=0.15, label='Mascara extra')

    handles, labels = axl1.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    axl1.legend(by_label.values(), by_label.keys(), loc='best')

    axl1.set_ylabel('Fluxo')
    axl1.set_title(f'Regiao manual {k}: [{ini:.4f}, {fim:.4f}]')
    axl1.grid(alpha=0.25)

    axl2.plot(tempo_local_det, fluxo_local_det, 'b.-', ms=3, lw=0.8, alpha=0.8, label='Flatten local')
    axl2.axhline(1.0, color='gray', ls='--', lw=1.0)
    axl2.set_xlabel('Tempo [BTJD dias]')
    axl2.set_ylabel('Fluxo detrended')
    axl2.legend(loc='best')
    axl2.grid(alpha=0.25)

    plt.tight_layout()
    plt.show()

print("\nPronto: os intervalos do CSV sao sempre excluidos do ajuste. Ajustes manuais so adicionam refinamentos.")
"""


AJUSTE: Polinomial com Segmentos Automaticos + Manuais
Ajuste automatico: 10 segmentos de tempo iguais.
Mascara usada no ajuste: intervalos do CSV (sempre excluidos).
Aplicando 5 ajuste(s) manual(is):
  OK [3883.005, 3883.4554] -> grau=4, sigma=1.5 (323 pontos)
  OK [3883.1409, 3883.7849] -> grau=4, sigma=2.5 (464 pontos)
  OK [3884.2828, 3884.6983] -> grau=3, sigma=3.5 (299 pontos)
  OK [3884.665, 3885.1138] -> grau=4, sigma=2.5 (324 pontos)
  OK [3885.1191, 3885.2983] -> grau=4, sigma=2.5 (129 pontos)

OK Ajuste concluido. Graficos de inspecao abaixo.


'\n\n\n# ============================================================\n# GRAFICO 3: ajustes manuais estilo recorte + flatten por regiao\n# ============================================================\nmanual_flatten_results = []\n\nfor k, cfg in enumerate(ajustes_flatten_manuais, start=1):\n    ini, fim = cfg["intervalo"]\n    idx_reg = (x2min >= ini) & (x2min <= fim)\n\n    if not np.any(idx_reg):\n        print(f"[AVISO] Ajuste local {k}: sem pontos no intervalo [{ini}, {fim}].")\n        continue\n\n    x_local = np.array(x2min[idx_reg], dtype=np.float64)\n    y_local = np.array(y2min[idx_reg], dtype=np.float64)\n\n    mascara_local = np.zeros(len(x_local), dtype=bool)\n\n    # Mascara base obrigatoria: intervalos do CSV que cruzam a regiao\n    intervalos_csv_regiao = []\n    for m_ini, m_fim in mascara_flares_list:\n        if (m_fim >= ini) and (m_ini <= fim):\n            intervalos_csv_regiao.append([m_ini, m_fim])\n\n    # Mascara extra opcional definida manualmente\n    inter

In [66]:


%matplotlib qt
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Gráfico 1: Dados originais + manchas
ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados Originais')
ax1.plot(t, modelo_manchas, 'r-', linewidth=2.5, label='Ajuste (Manchas)')
ax1.set_ylabel('Fluxo Normalizado', fontsize=14)
ax1.set_title('Passo 1: Ajuste de Manchas Estelares', fontsize=15, fontweight='bold')
ax1.legend(loc='upper right', fontsize=12)
ax1.grid(alpha=0.3)





In [67]:
# ============================================================
# PASSO 3: VISUALIZAR RESIDUAIS E SELECIONAR FLARES INTERATIVAMENTE
# ============================================================
print("\n" + "="*60)
print("VISUALIZAÇÃO: Residuais após manchas (para seleção de flares)")
print("="*60)

%matplotlib qt
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Gráfico 1: Dados originais + manchas
ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados Originais')
ax1.plot(t, modelo_manchas, 'r-', linewidth=2.5, label='Ajuste (Manchas)')
ax1.set_ylabel('Fluxo Normalizado', fontsize=14)
ax1.set_title('Passo 1: Ajuste de Manchas Estelares', fontsize=15, fontweight='bold')
ax1.legend(loc='upper right', fontsize=12)
ax1.grid(alpha=0.3)

# Gráfico 2: Residuais para detecção de flares
ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.7, label='Residual (original/manchas)')
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylabel('Fluxo Residual', fontsize=14)
ax2.set_xlabel('Tempo [BTJD dias]', fontsize=14)
ax2.set_title('Residuais para Seleção de Flares e Trânsitos', fontsize=15, fontweight='bold')
ax2.legend(loc='upper right', fontsize=12)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n→ Feche o gráfico e continue na próxima célula para inserir os intervalos de flares/trânsitos")



VISUALIZAÇÃO: Residuais após manchas (para seleção de flares)

→ Feche o gráfico e continue na próxima célula para inserir os intervalos de flares/trânsitos


In [ ]:

# ============================================================
# PASSO 5: SEGUNDA ITERAÇÃO - AJUSTE DE FLARES/TRÂNSITOS
# COM CONVERGÊNCIA (MELHORA RMS < 1%)
# ============================================================
print("\n" + "="*60)
print("SEGUNDA ITERAÇÃO: Ajuste de Flares/Trânsitos com Convergência")
print("="*60)

# Começa com modelo de manchas como baseline
modelo_flares = modelo_manchas.copy()
residual_anterior = residual_manchas.copy()
rms_inicial = np.sqrt(np.mean((residual_anterior - 1.0)**2))

convergencia_atingida = False
num_iteracoes_globais = 0
MAX_ITERACOES = 10  # Limite de segurança

while not convergencia_atingida and num_iteracoes_globais < MAX_ITERACOES:
    num_iteracoes_globais += 1
    print(f"\n--- Iteração Global {num_iteracoes_globais} ---")
    
    # Ajusta flares/trânsitos excluindo os intervalos definidos
    modelo_flares_novo = np.zeros_like(f)
    
    for seg_num, (seg_ini, seg_fim) in enumerate(segmentos):
        seg = (t >= seg_ini) & (t <= seg_fim)
        if not np.any(seg): continue
        
        t_seg = t[seg]
        f_seg = f[seg]
        
        # Centralizar tempo para evitar RankWarning
        t_mid = np.median(t_seg)
        t_shifted = t_seg - t_mid
        
        good = mask_good_flares[seg].copy()
        
        for iter_num in range(5):  # 5 iterações por segmento
            if good.sum() < (4 + 2): # Conferir se tem pontos para deg=4
                break
            
            coef = np.polyfit(t_shifted[good], f_seg[good], deg=4)
            
            # 1. Avalia APENAS onde tem dado bom
            modelo_good = np.polyval(coef, t_shifted[good])
            
            # 2. Passa uma reta suave cobrindo a região excluída
            modelo = np.interp(t_shifted, t_shifted[good], modelo_good)
            
            resid = f_seg - modelo
            
            clipped = sigma_clip(resid, sigma=2.5, maxiters=1)
            
            if clipped.mask is np.ma.nomask:
                new_good = good.copy()
            else:
                new_good = (~clipped.mask) & good
            
            good = new_good
        
        modelo_flares_novo[seg] = modelo
    
    # Calcula novo residual
    residual_novo = f / modelo_flares_novo
    rms_novo = np.sqrt(np.mean((residual_novo - 1.0)**2))
    
    # RMS anterior calculado apenas nos pontos bons para comparação justa
    rms_ant_check = np.sqrt(np.mean((residual_anterior - 1.0)**2))
    melhora_relativa = (1 - rms_novo / rms_ant_check) * 100
    
    print(f"  RMS anterior: {rms_ant_check:.6f}")
    print(f"  RMS novo:     {rms_novo:.6f}")
    print(f"  Melhora:      {melhora_relativa:.2f}%")
    
    if melhora_relativa < 1.0:
        print(f"\n✓ CONVERGÊNCIA ATINGIDA! (melhora < 1%)")
        convergencia_atingida = True
    else:
        modelo_flares = modelo_flares_novo.copy()
        residual_anterior = residual_novo.copy()

# Resultado final
tempo_trend = t
fluxo_trend = modelo_flares
fluxo_flat = f / modelo_flares
tempo_flat = t

print(f"\n" + "="*60)
print(f"✓ Detrending concluído!")
print(f"  Total de iterações globais: {num_iteracoes_globais}")
print(f"  RMS final: {np.sqrt(np.mean((residual_novo - 1.0)**2)):.6f}")
print(f"="*60)



SEGUNDA ITERAÇÃO: Ajuste de Flares/Trânsitos com Convergência

--- Iteração Global 1 ---
  RMS anterior: 0.002224
  RMS novo:     0.002208
  Melhora:      0.71%

✓ CONVERGÊNCIA ATINGIDA! (melhora < 1%)

✓ Detrending concluído!
  Total de iterações globais: 1
  RMS final: 0.002208


In [69]:
%matplotlib qt
import matplotlib.pyplot as plt

print("\n" + "="*60)
print("GRÁFICOS FINAIS: Comparação Original vs Detrended")
print("="*60)

fig, axes = plt.subplots(3, 1, figsize=(18, 12))

# --- Gráfico 1: Dados originais + Ajuste final ---
axes[0].plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados Originais')
axes[0].plot(tempo_trend, fluxo_trend, 'r-', linewidth=2.5, label='Ajuste Final (Manchas+Flares)')
axes[0].set_ylabel('Fluxo Normalizado', fontsize=14)
axes[0].set_title('Passo 1: Dados Originais + Tendência', fontsize=15, fontweight='bold')
axes[0].legend(loc='upper right', fontsize=12)
axes[0].grid(alpha=0.3)

# --- Gráfico 2: Residual detrended ---
axes[1].plot(tempo_flat, fluxo_flat, 'b.-', ms=1.5, lw=0.5, alpha=0.7)
axes[1].axhline(y=1.0, color='gray', linestyle='--', linewidth=1.5, alpha=0.7)
axes[1].fill_between(tempo_flat, 0.98, 1.02, alpha=0.2, color='green', label='Band ±2%')
axes[1].set_ylabel('Fluxo Normalizado', fontsize=14)
axes[1].set_title('Passo 2: Curva Detrended (Residuais)', fontsize=15, fontweight='bold')
axes[1].legend(loc='upper right', fontsize=12)
axes[1].grid(alpha=0.3)

# --- Gráfico 3: Zoom em região com flares ---
if len(mascara_flares_list) > 0:
    zoom_ini, zoom_fim = mascara_flares_list[0]
    zoom_mask = (t >= zoom_ini - 0.05) & (t <= zoom_fim + 0.05)
    
    axes[2].plot(t[zoom_mask], f[zoom_mask], 'ko-', ms=3, lw=1, alpha=0.7, label='Original')
    axes[2].plot(tempo_trend[zoom_mask], fluxo_trend[zoom_mask], 'r-', linewidth=2.5, label='Ajuste')
    
    # Marca a região do flare
    for ini, fim in mascara_flares_list:
        axes[2].axvspan(ini, fim, alpha=0.2, color='red', label='Flare/Trânsito (excluído)' if ini == mascara_flares_list[0][0] else '')
    
    axes[2].set_xlabel('Tempo [BTJD dias]', fontsize=14)
    axes[2].set_ylabel('Fluxo Normalizado', fontsize=14)
    axes[2].set_title('Zoom: Região com Flare/Trânsito', fontsize=15, fontweight='bold')
    axes[2].legend(loc='upper right', fontsize=12)
    axes[2].grid(alpha=0.3)
else:
    axes[2].text(0.5, 0.5, 'Nenhum flare/trânsito selecionado', 
                 ha='center', va='center', transform=axes[2].transAxes, fontsize=14)

plt.tight_layout()
plt.show()

print("\n✓ Gráficos plotados com sucesso!")



GRÁFICOS FINAIS: Comparação Original vs Detrended

✓ Gráficos plotados com sucesso!
